# Deepfake Detection That Shows Its Work
## Localizing Fake Facial Regions with Grad-CAM and Grad-CAM++

**Course:** AIN3002 — Deep Learning  
**Dataset:** Celeb-DF v2  
**Framework:** PyTorch + torchvision (T4 GPU on Colab)

---

### Full Pipeline

| # | Stage | Key outputs |
|---|-------|-------------|
| 1 | Dataset download & prep | Celeb-DF v2 video files |
| 2 | Face extraction (1 fps, MTCNN) | Aligned 224×224 face crops |
| 3 | EfficientNet-B0 fine-tuning | Trained binary real/fake classifier |
| 4 | Evaluation | Accuracy · AUC · Precision · Recall · Confusion Matrix |
| 5 | Grad-CAM & Grad-CAM++ | Per-image saliency heatmaps |
| 6 | Faithfulness evaluation | Insertion / Deletion AUC comparison |

> **Novel contribution:** Side-by-side quantitative faithfulness comparison of  
> Grad-CAM vs Grad-CAM++ using insertion/deletion pixel-perturbation metrics  
> on a deepfake detection task — linking *where* the network looks to *how  
> much* that region drives the prediction.


---
## Section 0 — Environment Setup


In [7]:
%%capture
!pip install facenet-pytorch==2.5.3 albumentations==1.3.1 scipy==1.11.4
print("Packages ready.")

In [8]:
import os, re, json, random, shutil, warnings, time
from pathlib import Path
from collections import defaultdict
from typing import List, Tuple, Dict, Optional

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from tqdm.notebook import tqdm
import scipy.ndimage as ndimage

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
import torchvision.transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_curve
)
from sklearn.model_selection import train_test_split
from facenet_pytorch import MTCNN

warnings.filterwarnings('ignore')
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [ ]:
# ── Central configuration ────────────────────────────────────────────────────
CONFIG = {
    # Paths
    'data_dir': '/kaggle/input/celeb-df2',
    'frames_dir':      '/kaggle/working/frames',
    'results_dir':     '/kaggle/working/results',
    'checkpoint_dir':  '/kaggle/working/checkpoints',

    # Face extraction
    'img_size':        224,
    'frames_fps':      1,           # frames per second to sample
    'max_frames_per_video': 30,     # cap per video to avoid class skew

    # Training
    'batch_size':      32,
    'num_epochs':      20,
    'lr':              1e-4,
    'weight_decay':    1e-5,
    'patience':        5,           # early-stopping patience
    'seed':            42,
    'num_workers':     2,

    # Model
    'num_classes':     2,
    'dropout':         0.4,

    # Explainability
    'n_gradcam_samples':  20,
    'n_insertion_steps':  100,

    # Set True to run without the real dataset (generates synthetic faces)
    'demo_mode':       False,
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])

# Create all output directories up-front
for d in [
    CONFIG['results_dir'],
    CONFIG['checkpoint_dir'],
    CONFIG['frames_dir'],
    f"{CONFIG['results_dir']}/training",
    f"{CONFIG['results_dir']}/evaluation",
    f"{CONFIG['results_dir']}/gradcam",
    f"{CONFIG['results_dir']}/faithfulness",
]:
    os.makedirs(d, exist_ok=True)

print("Directories ready.")

---
## Section 1 — Dataset Download & Preparation

### How to get Celeb-DF v2

Celeb-DF v2 is free for academic use. Follow these steps:

1. **Request access** at <https://github.com/yuezunli/celeb-deepfakeforensics>  
   Fill the Google Form — you will receive a Google Drive download link.

2. **Upload to your Google Drive** (e.g. `My Drive/datasets/Celeb-DF-v2/`).

3. **Mount Drive** and copy the folder:
   ```python
   from google.colab import drive
   drive.mount('/content/drive')
   !cp -r /content/drive/MyDrive/datasets/Celeb-DF-v2 /content/
   ```

Expected directory layout after extraction:
```
Celeb-DF-v2/
├── Celeb-real/           # 590 real celebrity videos
├── Celeb-synthesis/      # 5 639 fake celebrity videos
├── YouTube-real/         # 300 real YouTube videos
└── List_of_testing_videos.txt
```

> **Quick test** — set `CONFIG['demo_mode'] = True` to skip downloading  
> and run the full pipeline on synthetic placeholder images.


In [ ]:
print("Dataset ready!")

# ── Demo-mode synthetic dataset generator ────────────────────────────────────
def generate_demo_dataset(frames_dir: str, n_real: int = 300, n_fake: int = 700):
    """Create random synthetic face images so the full pipeline can be tested."""
    for label_name, n in [('real', n_real), ('fake', n_fake)]:
        folder = Path(frames_dir) / label_name
        folder.mkdir(parents=True, exist_ok=True)
        for i in range(n):
            img = np.random.randint(100, 200, (224, 224, 3), dtype=np.uint8)
            # Rough skin-tone ellipse to mimic a face
            cv2.ellipse(img, (112, 112), (70, 90), 0, 0, 360, (200, 175, 155), -1)
            # Add some per-class texture difference so the model can learn
            if label_name == 'fake':
                noise = np.random.randint(0, 80, img.shape, dtype=np.uint8)
                img = cv2.add(img, noise)
            cv2.imwrite(str(folder / f"demo_{i:05d}.jpg"), img)
    print(f"Demo dataset: {n_real} real + {n_fake} fake images in {frames_dir}")

# ── Dataset verification ──────────────────────────────────────────────────────
def verify_celebdf_structure(data_dir: str) -> bool:
    required = ['Celeb-real', 'Celeb-synthesis', 'YouTube-real',
                'List_of_testing_videos.txt']
    missing = [r for r in required if not os.path.exists(os.path.join(data_dir, r))]
    if missing:
        print(f"Missing from {data_dir}: {missing}")
        return False
    real_vids  = len(list(Path(data_dir, 'Celeb-real').glob('*.mp4')))
    real_vids += len(list(Path(data_dir, 'YouTube-real').glob('*.mp4')))
    fake_vids  = len(list(Path(data_dir, 'Celeb-synthesis').glob('*.mp4')))
    print(f"Celeb-DF v2 found — real: {real_vids}, fake: {fake_vids}")
    return True

if CONFIG['demo_mode']:
    print("[DEMO MODE] Generating synthetic dataset...")
    generate_demo_dataset(CONFIG['frames_dir'])
    SKIP_EXTRACTION = True   # jump directly to Section 3
else:
    ok = verify_celebdf_structure(CONFIG['data_dir'])
    if not ok:
        raise FileNotFoundError(
            f"Dataset not found at {CONFIG['data_dir']}. "
            "Follow the instructions above or set CONFIG['demo_mode']=True."
        )
    SKIP_EXTRACTION = False
    print("Dataset structure verified.")

---
## Section 2 — Face Extraction & Preprocessing

For each video we:
1. Sample **1 frame per second** with OpenCV.
2. Detect and align the dominant face with **MTCNN** (facenet-pytorch).
3. Crop and resize to **224 × 224** (EfficientNet-B0 input size).
4. Save to `frames/{real,fake}/` — each file is named `{video_id}_{frame:04d}.jpg`.

Frames without a detectable face are skipped.


In [ ]:
if not SKIP_EXTRACTION:
    # Initialise MTCNN face detector (runs on GPU if available)
    face_detector = MTCNN(
        image_size=CONFIG['img_size'],
        margin=20,
        min_face_size=60,
        thresholds=[0.6, 0.7, 0.7],
        factor=0.709,
        post_process=False,   # return uint8 [0,255] crops
        device=DEVICE,
        keep_all=False,       # return only the largest face
    )
    print("MTCNN detector ready.")

In [ ]:
def extract_faces_from_video(
    video_path: str,
    out_dir: str,
    video_id: str,
    fps_target: int = 1,
    max_frames: int = 30,
    img_size: int = 224,
) -> int:
    """Extract aligned face crops from a video at fps_target frames/sec.

    Returns the number of faces successfully saved.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0

    fps_orig = cap.get(cv2.CAP_PROP_FPS)
    if fps_orig <= 0:
        fps_orig = 30.0
    frame_interval = max(1, int(round(fps_orig / fps_target)))

    os.makedirs(out_dir, exist_ok=True)
    saved = 0
    frame_idx = 0

    while saved < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            # Convert BGR → RGB for MTCNN
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(rgb)

            # Detect and crop face
            face_crop = face_detector(pil_img)  # Tensor [3, H, W] or None
            if face_crop is not None:
                # MTCNN returns float32 in [0,255] when post_process=False
                face_np = face_crop.permute(1, 2, 0).byte().numpy()
                face_np = cv2.resize(face_np, (img_size, img_size))
                out_path = os.path.join(out_dir, f"{video_id}_{saved:04d}.jpg")
                cv2.imwrite(out_path, cv2.cvtColor(face_np, cv2.COLOR_RGB2BGR))
                saved += 1

        frame_idx += 1

    cap.release()
    return saved

In [ ]:
if not SKIP_EXTRACTION:
    data_dir = Path(CONFIG['data_dir'])

    # Collect (video_path, label_name) pairs
    video_pairs: List[Tuple[Path, str]] = []
    for mp4 in sorted(data_dir.glob('Celeb-real/*.mp4')):
        video_pairs.append((mp4, 'real'))
    for mp4 in sorted(data_dir.glob('YouTube-real/*.mp4')):
        video_pairs.append((mp4, 'real'))
    for mp4 in sorted(data_dir.glob('Celeb-synthesis/*.mp4')):
        video_pairs.append((mp4, 'fake'))

    print(f"Total videos to process: {len(video_pairs)}")

    total_saved = 0
    for vid_path, label in tqdm(video_pairs, desc='Extracting faces'):
        out_dir = os.path.join(CONFIG['frames_dir'], label)
        n = extract_faces_from_video(
            str(vid_path),
            out_dir,
            video_id=vid_path.stem,
            fps_target=CONFIG['frames_fps'],
            max_frames=CONFIG['max_frames_per_video'],
            img_size=CONFIG['img_size'],
        )
        total_saved += n

    n_real = len(list(Path(CONFIG['frames_dir'], 'real').glob('*.jpg')))
    n_fake = len(list(Path(CONFIG['frames_dir'], 'fake').glob('*.jpg')))
    print(f"Extraction complete — real: {n_real}, fake: {n_fake}, total: {total_saved}")

---
## Section 3 — PyTorch Dataset & DataLoaders

* **Train augmentation** — RandomHorizontalFlip, ColorJitter, RandomRotation, Gaussian blur.
* **Val/Test** — only resize + normalize (no augmentation).
* **WeightedRandomSampler** balances the real/fake class ratio in every training batch.


In [ ]:
class DeepfakeDataset(Dataset):
    """Loads pre-extracted face crops from frames/real and frames/fake."""

    def __init__(self, image_paths: List[str], labels: List[int], transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


# ImageNet mean/std — used for both train and eval
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_transforms(split: str, img_size: int = 224):
    if split == 'train':
        return T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
            T.RandomRotation(degrees=10),
            T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:  # val / test
        return T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])


print("Dataset class and transforms defined.")

In [ ]:
# ── Collect all image paths and labels ───────────────────────────────────────
frames_root = Path(CONFIG['frames_dir'])

all_paths, all_labels = [], []
for label_name, label_id in [('real', 0), ('fake', 1)]:
    folder = frames_root / label_name
    paths = sorted(folder.glob('*.jpg'))
    all_paths.extend([str(p) for p in paths])
    all_labels.extend([label_id] * len(paths))

print(f"Total images: {len(all_paths)}  "
      f"(real={all_labels.count(0)}, fake={all_labels.count(1)})")

# ── Stratified split: 70% train / 15% val / 15% test ─────────────────────────
train_paths, tmp_paths, train_labels, tmp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30,
    stratify=all_labels, random_state=CONFIG['seed']
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    tmp_paths, tmp_labels, test_size=0.50,
    stratify=tmp_labels, random_state=CONFIG['seed']
)

print(f"Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")

# ── Datasets ──────────────────────────────────────────────────────────────────
train_ds = DeepfakeDataset(train_paths, train_labels, get_transforms('train', CONFIG['img_size']))
val_ds   = DeepfakeDataset(val_paths,   val_labels,   get_transforms('val',   CONFIG['img_size']))
test_ds  = DeepfakeDataset(test_paths,  test_labels,  get_transforms('test',  CONFIG['img_size']))

# ── WeightedRandomSampler to handle class imbalance ──────────────────────────
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts.astype(np.float32)
sample_weights = [class_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                          sampler=sampler, num_workers=CONFIG['num_workers'],
                          pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=CONFIG['num_workers'],
                          pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=CONFIG['num_workers'],
                          pin_memory=True)

print(f"DataLoaders created (batch size {CONFIG['batch_size']}).")

---
## Section 4 — EfficientNet-B0 Model

We use **EfficientNet-B0** pretrained on ImageNet (torchvision).  
The first 4 feature blocks are frozen; blocks 4–8 and the classifier are fine-tuned.  
The final linear layer is replaced with a 2-class head for real/fake classification.


In [ ]:
class DeepfakeDetector(nn.Module):
    """EfficientNet-B0 fine-tuned for binary deepfake detection."""

    def __init__(self, num_classes: int = 2, dropout: float = 0.4):
        super().__init__()
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        self.backbone = efficientnet_b0(weights=weights)

        # Freeze the first 4 feature blocks (low-level edge / texture detectors)
        for i, block in enumerate(self.backbone.features):
            if i < 4:
                for param in block.parameters():
                    param.requires_grad = False

        # Replace the classifier head
        in_features = self.backbone.classifier[1].in_features  # 1280
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)

    def get_gradcam_target_layer(self) -> nn.Module:
        """Return the last ConvNormActivation block (features[8]) for Grad-CAM."""
        return self.backbone.features[-1]


model = DeepfakeDetector(
    num_classes=CONFIG['num_classes'],
    dropout=CONFIG['dropout']
).to(DEVICE)

# Count trainable parameters
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters — total: {total:,}  |  trainable: {trainable:,}")

---
## Section 5 — Training

* **Optimizer:** Adam with weight decay `1e-5`
* **LR Schedule:** CosineAnnealingLR (`T_max = num_epochs`)
* **Early stopping:** restore best weights when val AUC stops improving (patience = 5)
* **Checkpointing:** saves best model to `checkpoints/best_model.pth`


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['num_epochs'], eta_min=1e-6
)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total   += imgs.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)
        probs = torch.softmax(logits, dim=1)[:, 1]  # prob of fake
        correct += (logits.argmax(1) == labels).sum().item()
        total   += imgs.size(0)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    avg_loss = running_loss / total
    acc = correct / total
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    return avg_loss, acc, auc


print("Optimizer, scheduler, and training functions ready.")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
history = defaultdict(list)
best_val_auc = 0.0
patience_counter = 0
best_ckpt = os.path.join(CONFIG['checkpoint_dir'], 'best_model.pth')

for epoch in range(1, CONFIG['num_epochs'] + 1):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss,   val_acc,   val_auc = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    elapsed = time.time() - t0
    print(f"Epoch {epoch:02d}/{CONFIG['num_epochs']}  "
          f"train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
          f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  val_auc={val_auc:.4f}  "
          f"({elapsed:.1f}s)")

    # Save checkpoint if validation AUC improved
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        torch.save(model.state_dict(), best_ckpt)
        print(f"  ↳ New best val AUC = {best_val_auc:.4f} — checkpoint saved.")
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['patience']:
            print(f"Early stopping at epoch {epoch}.")
            break

print(f"\nTraining finished. Best val AUC: {best_val_auc:.4f}")

In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'],   label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, history['val_auc'], color='green', label='Val AUC')
axes[2].set_title('Validation AUC-ROC'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
save_path = f"{CONFIG['results_dir']}/training/training_curves.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

---
## Section 6 — Evaluation Metrics

Load the best checkpoint and run inference on the held-out test set.  
Metrics reported: **Accuracy, AUC-ROC, Precision, Recall, F1**.  
Figures saved: Confusion Matrix, ROC Curve.


In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()
print(f"Loaded best model from {best_ckpt}")

# Run test inference
all_preds, all_probs_test, all_true = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Test inference'):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_probs_test.extend(probs)
        all_true.extend(labels.numpy())

all_preds      = np.array(all_preds)
all_probs_test = np.array(all_probs_test)
all_true       = np.array(all_true)

In [ ]:
# ── Compute metrics ───────────────────────────────────────────────────────────
test_metrics = {
    'accuracy':  accuracy_score(all_true, all_preds),
    'auc_roc':   roc_auc_score(all_true, all_probs_test),
    'precision': precision_score(all_true, all_preds, zero_division=0),
    'recall':    recall_score(all_true, all_preds, zero_division=0),
    'f1':        f1_score(all_true, all_preds, zero_division=0),
}

print("\n=== Test-set Results ===")
for k, v in test_metrics.items():
    print(f"  {k:<12}: {v:.4f}")

with open(f"{CONFIG['results_dir']}/evaluation/metrics.json", 'w') as f:
    json.dump(test_metrics, f, indent=2)
print("Metrics saved.")

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_true, all_preds)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Real', 'Fake'])
ax.set_yticklabels(['Real', 'Fake'])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix')

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)

plt.tight_layout()
save_path = f"{CONFIG['results_dir']}/evaluation/confusion_matrix.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(all_true, all_probs_test)
auc_val = test_metrics['auc_roc']

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, lw=2, label=f'AUC = {auc_val:.4f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
save_path = f"{CONFIG['results_dir']}/evaluation/roc_curve.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

---
## Section 7 — Grad-CAM & Grad-CAM++ (Novel Contribution)

Both methods hook into **`backbone.features[-1]`** (the final 1280-channel
ConvNormActivation block of EfficientNet-B0, producing 7×7 feature maps for
224×224 input).

### Grad-CAM (Selvaraju et al., 2017)
$$
L^c_{\text{Grad-CAM}} = \text{ReLU}\!\left(
  \sum_k \underbrace{\frac{1}{Z}\sum_{i,j}\frac{\partial y^c}{\partial A^k_{ij}}}_{\alpha^c_k} A^k
\right)
$$

### Grad-CAM++ (Chattopadhay et al., 2018)
Uses second-order gradient information to compute per-pixel importance weights:
$$
\alpha^c_k = \frac{\dfrac{\partial^2 y^c}{\partial (A^k_{ij})^2}}
{2\,\dfrac{\partial^2 y^c}{\partial (A^k_{ij})^2}
 + \displaystyle\sum_{a,b} A^k_{ab}\,\dfrac{\partial^3 y^c}{\partial (A^k_{ij})^3}}
$$

This lets Grad-CAM++ better highlight **multiple** discriminative regions
(e.g. both eyes, mouth, and forehead simultaneously).


In [ ]:
class GradCAM:
    """Grad-CAM implementation (Selvaraju et al., ICCV 2017)."""

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self._activations: Optional[torch.Tensor] = None
        self._gradients:   Optional[torch.Tensor] = None
        self._hooks = [
            target_layer.register_forward_hook(self._fwd_hook),
            target_layer.register_full_backward_hook(self._bwd_hook),
        ]

    def _fwd_hook(self, module, inp, output):
        self._activations = output.detach()

    def _bwd_hook(self, module, grad_in, grad_out):
        self._gradients = grad_out[0].detach()

    def __call__(
        self,
        x: torch.Tensor,          # [1, 3, H, W] — preprocessed input
        class_idx: Optional[int] = None,
    ) -> np.ndarray:               # [H, W] saliency map in [0, 1]

        self.model.eval()
        x = x.clone().requires_grad_(True)
        logits = self.model(x)

        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())

        self.model.zero_grad()
        logits[0, class_idx].backward(retain_graph=True)

        # Global-average-pool the gradients → per-channel weight
        weights = self._gradients.mean(dim=(2, 3), keepdim=True)  # [1,C,1,1]

        # Weighted sum of activation maps, then ReLU
        cam = (weights * self._activations).sum(dim=1, keepdim=True)  # [1,1,h,w]
        cam = F.relu(cam)

        # Upsample to input resolution
        cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()

        # Normalize to [0, 1]
        lo, hi = cam.min(), cam.max()
        if hi > lo:
            cam = (cam - lo) / (hi - lo)
        return cam

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()


print("GradCAM class defined.")

In [ ]:
class GradCAMPlusPlus:
    """Grad-CAM++ implementation (Chattopadhay et al., WACV 2018).

    Key difference: uses second-order gradient information to weight each
    spatial location of the gradient map individually, rather than globally
    averaging.  Better at highlighting *multiple* discriminative regions.
    """

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self._activations: Optional[torch.Tensor] = None
        self._gradients:   Optional[torch.Tensor] = None
        self._hooks = [
            target_layer.register_forward_hook(self._fwd_hook),
            target_layer.register_full_backward_hook(self._bwd_hook),
        ]

    def _fwd_hook(self, module, inp, output):
        self._activations = output.detach()

    def _bwd_hook(self, module, grad_in, grad_out):
        self._gradients = grad_out[0].detach()

    def __call__(
        self,
        x: torch.Tensor,
        class_idx: Optional[int] = None,
    ) -> np.ndarray:

        self.model.eval()
        x = x.clone().requires_grad_(True)
        logits = self.model(x)

        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())

        self.model.zero_grad()
        logits[0, class_idx].backward(retain_graph=True)

        grads = self._gradients      # [1, C, h, w]
        acts  = self._activations    # [1, C, h, w]

        # ── Grad-CAM++ alpha computation ─────────────────────────────────────
        grads_sq = grads ** 2
        grads_cu = grads ** 3
        eps = 1e-8

        # Global spatial sum of activations (for the denominator term)
        global_sum = acts.sum(dim=(2, 3), keepdim=True)  # [1, C, 1, 1]

        # alpha_ij = grad² / (2*grad² + global_sum * grad³ + eps)
        alpha = grads_sq / (2.0 * grads_sq + global_sum * grads_cu + eps)

        # Zero out positions where gradient is zero (avoid 0/0 artefacts)
        alpha = torch.where(grads != 0, alpha, torch.zeros_like(alpha))

        # Weight = spatial sum of (alpha * ReLU(gradient))
        weights = (alpha * F.relu(grads)).sum(dim=(2, 3), keepdim=True)

        # Weighted activation sum + ReLU
        cam = F.relu((weights * acts).sum(dim=1, keepdim=True))

        cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()

        lo, hi = cam.min(), cam.max()
        if hi > lo:
            cam = (cam - lo) / (hi - lo)
        return cam

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()


print("GradCAMPlusPlus class defined.")

In [ ]:
# ── Visualization helpers ─────────────────────────────────────────────────────

# Jet-like colormap (same as OpenCV's COLORMAP_JET)
_JET = LinearSegmentedColormap.from_list(
    'jet_custom',
    [(0,'#000080'), (0.25,'#0080FF'), (0.5,'#00FF80'),
     (0.75,'#FFFF00'), (1.0,'#FF0000')]
)

def denormalize(tensor: torch.Tensor) -> np.ndarray:
    """Convert a normalized ImageNet tensor [3,H,W] to uint8 HxWx3 numpy."""
    mean = np.array(IMAGENET_MEAN, dtype=np.float32)[:, None, None]
    std  = np.array(IMAGENET_STD,  dtype=np.float32)[:, None, None]
    img  = tensor.cpu().numpy() * std + mean
    img  = np.clip(img * 255, 0, 255).astype(np.uint8)
    return img.transpose(1, 2, 0)  # HxWx3


def overlay_heatmap(
    image_np: np.ndarray,      # HxWx3 uint8
    cam: np.ndarray,           # HxW float [0,1]
    alpha: float = 0.45,
) -> np.ndarray:               # HxWx3 uint8
    """Blend a saliency map onto the original image."""
    heatmap = (_JET(cam)[:, :, :3] * 255).astype(np.uint8)  # HxWx3
    overlay = (alpha * heatmap + (1 - alpha) * image_np).astype(np.uint8)
    return overlay


def visualize_comparison(
    image_tensor: torch.Tensor,   # [1, 3, H, W]
    cam_gc:  np.ndarray,
    cam_gcpp: np.ndarray,
    true_label: int,
    pred_label: int,
    pred_prob:  float,
    sample_idx: int,
    save_dir: str,
):
    """Save a 4-panel figure: original | Grad-CAM | Grad-CAM++ | diff map."""
    img_np = denormalize(image_tensor.squeeze(0))

    overlay_gc   = overlay_heatmap(img_np, cam_gc)
    overlay_gcpp = overlay_heatmap(img_np, cam_gcpp)
    diff         = np.abs(cam_gcpp - cam_gc)   # highlight disagreements

    label_names = {0: 'Real', 1: 'Fake'}
    title_color = 'green' if true_label == pred_label else 'red'

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    panels = [
        (img_np,       'Original'),
        (overlay_gc,   'Grad-CAM'),
        (overlay_gcpp, 'Grad-CAM++'),
        (diff,         'Difference |GC++ − GC|'),
    ]
    for ax, (panel, title) in zip(axes, panels):
        if panel.ndim == 2:  # diff map
            ax.imshow(panel, cmap='hot', vmin=0, vmax=1)
        else:
            ax.imshow(panel)
        ax.set_title(title, fontsize=10)
        ax.axis('off')

    suptitle = (f"Sample {sample_idx}  |  True: {label_names[true_label]}  "
                f"Pred: {label_names[pred_label]} ({pred_prob:.2f})")
    fig.suptitle(suptitle, fontsize=12, color=title_color, fontweight='bold')

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"sample_{sample_idx:03d}_comparison.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return save_path


print("Visualization helpers ready.")

In [ ]:
# ── Instantiate Grad-CAM and Grad-CAM++ ──────────────────────────────────────
target_layer = model.get_gradcam_target_layer()
gradcam    = GradCAM(model, target_layer)
gradcam_pp = GradCAMPlusPlus(model, target_layer)

gradcam_dir = f"{CONFIG['results_dir']}/gradcam"

# Store CAMs for later faithfulness evaluation
gradcam_samples   = []   # list of dicts
sample_count      = 0
n_target          = CONFIG['n_gradcam_samples']

# Sample from test set (select first n_target images)
for imgs, labels in test_loader:
    for i in range(imgs.size(0)):
        if sample_count >= n_target:
            break

        img_t  = imgs[i:i+1].to(DEVICE)   # [1,3,H,W]
        true_l = int(labels[i].item())

        with torch.no_grad():
            logit = model(img_t)
        prob     = torch.softmax(logit, dim=1)[0, 1].item()
        pred_l   = int(logit.argmax(dim=1).item())

        cam_gc   = gradcam(img_t,    class_idx=pred_l)
        cam_gcpp = gradcam_pp(img_t, class_idx=pred_l)

        path = visualize_comparison(
            img_t, cam_gc, cam_gcpp,
            true_label=true_l, pred_label=pred_l, pred_prob=prob,
            sample_idx=sample_count, save_dir=gradcam_dir
        )

        gradcam_samples.append({
            'img_tensor': img_t.cpu(),
            'cam_gc':     cam_gc,
            'cam_gcpp':   cam_gcpp,
            'true':       true_l,
            'pred':       pred_l,
            'prob':       prob,
        })
        sample_count += 1

    if sample_count >= n_target:
        break

print(f"Generated {sample_count} Grad-CAM comparison figures in {gradcam_dir}")

In [ ]:
# ── Display a 2-row grid: Grad-CAM (top) vs Grad-CAM++ (bottom) ──────────────
n_show = min(8, len(gradcam_samples))

fig, axes = plt.subplots(3, n_show, figsize=(3 * n_show, 10))

for col, sample in enumerate(gradcam_samples[:n_show]):
    img_np = denormalize(sample['img_tensor'].squeeze(0))
    axes[0, col].imshow(img_np)
    axes[0, col].set_title(f"{'Real' if sample['true']==0 else 'Fake'}", fontsize=8)
    axes[0, col].axis('off')

    axes[1, col].imshow(overlay_heatmap(img_np, sample['cam_gc']))
    axes[1, col].axis('off')

    axes[2, col].imshow(overlay_heatmap(img_np, sample['cam_gcpp']))
    axes[2, col].axis('off')

axes[0, 0].set_ylabel('Original',   rotation=0, labelpad=60, va='center', fontsize=10)
axes[1, 0].set_ylabel('Grad-CAM',   rotation=0, labelpad=60, va='center', fontsize=10)
axes[2, 0].set_ylabel('Grad-CAM++', rotation=0, labelpad=60, va='center', fontsize=10)

plt.suptitle('Grad-CAM vs Grad-CAM++ Heatmaps', fontsize=13, fontweight='bold')
plt.tight_layout()
save_path = f"{CONFIG['results_dir']}/gradcam/grid_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Grid saved: {save_path}")

---
## Section 8 — Faithfulness Evaluation: Insertion & Deletion

A saliency map is **faithful** if regions it highlights actually drive the model's
prediction. We measure this with two pixel-perturbation protocols:

| Metric | Procedure | Good map → |
|--------|-----------|------------|
| **Insertion** | Start from a blurred image; progressively *reveal* pixels ordered by importance (most → least). Measure model confidence at each step. | AUC rises quickly → **higher is better** |
| **Deletion** | Start from the original; progressively *remove* pixels ordered by importance. Measure model confidence at each step. | AUC drops quickly → **lower is better** |

We compute both curves for every test sample and both methods, then compare
Grad-CAM vs Grad-CAM++ on mean Insertion-AUC and Deletion-AUC.


In [ ]:
def gaussian_blur_tensor(img_tensor: torch.Tensor, sigma: float = 10.0) -> torch.Tensor:
    """Apply channel-wise Gaussian blur to a [1,3,H,W] tensor (CPU or GPU)."""
    img_np = img_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()  # HxWx3
    blurred = ndimage.gaussian_filter(img_np, sigma=[sigma, sigma, 0])
    return torch.tensor(blurred, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)


def insertion_deletion_auc(
    model:       nn.Module,
    img_tensor:  torch.Tensor,   # [1, 3, H, W] — device tensor
    saliency:    np.ndarray,     # [H, W] float in [0,1]
    n_steps:     int = 100,
    device:      torch.device = DEVICE,
) -> Dict:
    """Compute insertion and deletion AUC for a single image/saliency pair."""
    model.eval()
    H, W = saliency.shape
    n_pixels = H * W

    # Descending order of pixel importance
    order = np.argsort(saliency.flatten())[::-1].copy()  # most → least important

    # Blurred baseline for the insertion metric
    blurred = gaussian_blur_tensor(img_tensor).to(device)

    step_size  = max(1, n_pixels // n_steps)
    ins_scores, del_scores, pcts = [], [], []

    for step in range(0, n_pixels, step_size):
        pcts.append(step / n_pixels)

        # Binary mask: 1 at the top-`step` most important pixels
        mask_flat      = np.zeros(n_pixels, dtype=np.float32)
        mask_flat[order[:step]] = 1.0
        mask = torch.tensor(
            mask_flat.reshape(1, 1, H, W), dtype=torch.float32, device=device
        )  # broadcast over channels

        # Insertion: reveal top pixels on blurred background
        ins_img = blurred * (1 - mask) + img_tensor * mask
        # Deletion: remove top pixels (replace with blurred)
        del_img = img_tensor * (1 - mask) + blurred * mask

        with torch.no_grad():
            ins_prob = torch.softmax(model(ins_img), dim=1)[0, 1].item()
            del_prob = torch.softmax(model(del_img), dim=1)[0, 1].item()

        ins_scores.append(ins_prob)
        del_scores.append(del_prob)

    ins_auc = float(np.trapz(ins_scores, pcts))
    del_auc = float(np.trapz(del_scores, pcts))

    return {
        'percentages':    pcts,
        'insertion_scores': ins_scores,
        'deletion_scores':  del_scores,
        'insertion_auc':  ins_auc,
        'deletion_auc':   del_auc,
    }


print("insertion_deletion_auc() defined.")

In [ ]:
# ── Run faithfulness evaluation on all Grad-CAM samples ──────────────────────
faith_results = {'gradcam': [], 'gradcam_pp': []}

for idx, sample in enumerate(tqdm(gradcam_samples, desc='Faithfulness eval')):
    img_t = sample['img_tensor'].to(DEVICE)

    res_gc   = insertion_deletion_auc(
        model, img_t, sample['cam_gc'],
        n_steps=CONFIG['n_insertion_steps']
    )
    res_gcpp = insertion_deletion_auc(
        model, img_t, sample['cam_gcpp'],
        n_steps=CONFIG['n_insertion_steps']
    )

    faith_results['gradcam'].append(res_gc)
    faith_results['gradcam_pp'].append(res_gcpp)

# ── Aggregate ─────────────────────────────────────────────────────────────────
def mean_metric(results, key):
    return float(np.mean([r[key] for r in results]))

summary_faith = {
    'GradCAM':    {'insertion_auc': mean_metric(faith_results['gradcam'],    'insertion_auc'),
                   'deletion_auc':  mean_metric(faith_results['gradcam'],    'deletion_auc')},
    'GradCAMpp':  {'insertion_auc': mean_metric(faith_results['gradcam_pp'], 'insertion_auc'),
                   'deletion_auc':  mean_metric(faith_results['gradcam_pp'], 'deletion_auc')},
}

print("\n=== Faithfulness Metrics (mean over test samples) ===")
for method, metrics in summary_faith.items():
    print(f"  {method:<12}  Insertion AUC = {metrics['insertion_auc']:.4f}  "
          f"Deletion AUC = {metrics['deletion_auc']:.4f}")

faith_path = f"{CONFIG['results_dir']}/faithfulness/faithfulness_metrics.json"
with open(faith_path, 'w') as f:
    json.dump(summary_faith, f, indent=2)
print(f"Saved: {faith_path}")

In [ ]:
# ── Plot mean Insertion and Deletion curves ───────────────────────────────────
def mean_curve(results, key):
    curves = np.array([r[key] for r in results])
    return curves.mean(axis=0), curves.std(axis=0)

pcts = faith_results['gradcam'][0]['percentages']

gc_ins_mean,   gc_ins_std   = mean_curve(faith_results['gradcam'],    'insertion_scores')
gcpp_ins_mean, gcpp_ins_std = mean_curve(faith_results['gradcam_pp'], 'insertion_scores')
gc_del_mean,   gc_del_std   = mean_curve(faith_results['gradcam'],    'deletion_scores')
gcpp_del_mean, gcpp_del_std = mean_curve(faith_results['gradcam_pp'], 'deletion_scores')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# -- Insertion --
ax1.plot(pcts, gc_ins_mean,   color='steelblue', lw=2,
         label=f"Grad-CAM   (AUC={summary_faith['GradCAM']['insertion_auc']:.3f})")
ax1.fill_between(pcts, gc_ins_mean - gc_ins_std, gc_ins_mean + gc_ins_std,
                 color='steelblue', alpha=0.2)
ax1.plot(pcts, gcpp_ins_mean, color='darkorange', lw=2,
         label=f"Grad-CAM++ (AUC={summary_faith['GradCAMpp']['insertion_auc']:.3f})")
ax1.fill_between(pcts, gcpp_ins_mean - gcpp_ins_std, gcpp_ins_mean + gcpp_ins_std,
                 color='darkorange', alpha=0.2)
ax1.set_title('Insertion (higher AUC = more faithful)', fontweight='bold')
ax1.set_xlabel('Fraction of pixels revealed')
ax1.set_ylabel('P(fake)')
ax1.legend(); ax1.grid(True, alpha=0.3)

# -- Deletion --
ax2.plot(pcts, gc_del_mean,   color='steelblue', lw=2,
         label=f"Grad-CAM   (AUC={summary_faith['GradCAM']['deletion_auc']:.3f})")
ax2.fill_between(pcts, gc_del_mean - gc_del_std, gc_del_mean + gc_del_std,
                 color='steelblue', alpha=0.2)
ax2.plot(pcts, gcpp_del_mean, color='darkorange', lw=2,
         label=f"Grad-CAM++ (AUC={summary_faith['GradCAMpp']['deletion_auc']:.3f})")
ax2.fill_between(pcts, gcpp_del_mean - gcpp_del_std, gcpp_del_mean + gcpp_del_std,
                 color='darkorange', alpha=0.2)
ax2.set_title('Deletion (lower AUC = more faithful)', fontweight='bold')
ax2.set_xlabel('Fraction of pixels removed')
ax2.set_ylabel('P(fake)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Faithfulness Evaluation: Insertion vs Deletion', fontsize=13, fontweight='bold')
plt.tight_layout()
save_path = f"{CONFIG['results_dir']}/faithfulness/insertion_deletion_curves.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

In [ ]:
# ── Bar chart comparing faithfulness AUC side-by-side ────────────────────────
methods     = ['Grad-CAM', 'Grad-CAM++']
ins_aucs    = [summary_faith['GradCAM']['insertion_auc'],
               summary_faith['GradCAMpp']['insertion_auc']]
del_aucs    = [summary_faith['GradCAM']['deletion_auc'],
               summary_faith['GradCAMpp']['deletion_auc']]

x = np.arange(len(methods))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
bars1 = ax.bar(x - width/2, ins_aucs, width, label='Insertion AUC (↑ better)',
               color=['steelblue', 'darkorange'])
bars2 = ax.bar(x + width/2, del_aucs, width, label='Deletion AUC (↓ better)',
               color=['steelblue', 'darkorange'], alpha=0.5, hatch='//')

ax.set_xticks(x); ax.set_xticklabels(methods)
ax.set_ylabel('AUC'); ax.set_title('Faithfulness Comparison', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
save_path = f"{CONFIG['results_dir']}/faithfulness/faithfulness_bar.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

---
## Section 9 — Results Summary


In [ ]:
# ── Collect and print the full results summary ────────────────────────────────
full_summary = {
    'classification': test_metrics,
    'faithfulness':   summary_faith,
}

summary_path = f"{CONFIG['results_dir']}/full_summary.json"
with open(summary_path, 'w') as f:
    json.dump(full_summary, f, indent=2)

print("="*55)
print(" FINAL RESULTS SUMMARY")
print("="*55)

print("\n[Classification — Test Set]")
for k, v in test_metrics.items():
    print(f"  {k:<12}: {v:.4f}")

print("\n[Faithfulness — Mean over Grad-CAM test samples]")
print(f"  {'Method':<14}  {'Insertion AUC':>15}  {'Deletion AUC':>13}")
print(f"  {'-'*45}")
for method, vals in summary_faith.items():
    print(f"  {method:<14}  {vals['insertion_auc']:>15.4f}  {vals['deletion_auc']:>13.4f}")

print(f"\nAll results saved to: {CONFIG['results_dir']}/")
print("\nResult files:")
for root, dirs, files in os.walk(CONFIG['results_dir']):
    level = root.replace(CONFIG['results_dir'], '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

In [ ]:
# ── Zip results folder for easy download ─────────────────────────────────────
import zipfile

zip_path = '/kaggle/working/results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(CONFIG['results_dir']):
        for file in files:
            file_path = os.path.join(root, file)
            arcname   = os.path.relpath(file_path, '/kaggle/working')
            zf.write(file_path, arcname)

print(f"Results zipped to {zip_path}")
print("Download via: Files panel (left sidebar) or run:")
print("  from google.colab import files; files.download('/content/results.zip')")